# Document Loading & Text Extraction

# 01. Document Loading and Text Extraction

## Objective

Understand how different document formats expose their textual content and investigate the extraction methods required to process them.

## Document Formats

Three document formats are investigated:

1. Plain Text (TXT)
2. Markdown (MD)
3. Portable Document Format (PDF)

## Extraction Methods

| Format | Extraction Method |
|---|---|
| TXT | Python file reading |
| Markdown | Python file reading |
| PDF | PyMuPDF page-level extraction |

## Experimental Procedure

1. Define a common source text.
2. Generate controlled document fixtures.
3. Extract text from each document.
4. Preserve individual PDF page boundaries.
5. Calculate character, word, and line counts.
6. Compare the extracted content with the original source.
7. Inspect differences introduced by document formatting.

## Evaluation Metrics

- Character count
- Word count
- Line count
- Extraction success
- Exact text equality
- Normalized text equality
- PDF page count

## Limitations

The generated documents are controlled experimental fixtures.

The results demonstrate basic document extraction behavior and do not represent extraction quality across arbitrary PDFs, scanned documents, complex tables, or multi-column layouts.

In [ ]:
from pathlib import Path
import json
import re

import pymupdf


# ============================================================
# 1. Project Paths
# ============================================================

PROJECT_ROOT = Path.cwd()

EXPERIMENT_DIR = PROJECT_ROOT / "data" / "experiments" / "day1_fixtures"

EXPERIMENT_DIR.mkdir(parents=True, exist_ok=True)


# ============================================================
# 2. Define Controlled Source Content
# ============================================================

SOURCE_PARAGRAPHS = [
    "HADOTA Local AI Assistant",
    "HADOTA is a local AI assistant powered by Qwen through Ollama.",
    "The application uses FastAPI as its backend framework.",
    "The system supports conversation management, structured outputs, "
    "and local language model inference.",
    "The project is being extended with Retrieval-Augmented Generation.",
    "RAG enables the assistant to retrieve relevant information "
    "from external knowledge sources before generating an answer.",
]

SOURCE_TEXT = "\n\n".join(SOURCE_PARAGRAPHS)


# ============================================================
# 3. Define Document Paths
# ============================================================

TXT_PATH = EXPERIMENT_DIR / "sample.txt"
MD_PATH = EXPERIMENT_DIR / "sample.md"
PDF_PATH = EXPERIMENT_DIR / "sample.pdf"


# ============================================================
# 4. Generate TXT Document
# ============================================================

TXT_PATH.write_text(SOURCE_TEXT, encoding="utf-8")


# ============================================================
# 5. Generate Markdown Document
# ============================================================

MARKDOWN_CONTENT = "# HADOTA Local AI Assistant\n\n" + SOURCE_TEXT

MD_PATH.write_text(MARKDOWN_CONTENT, encoding="utf-8")


# ============================================================
# 6. Generate a Two-Page PDF
# ============================================================

pdf_document = pymupdf.open()

pdf_pages_content = [
    "\n\n".join(SOURCE_PARAGRAPHS[:3]),
    "\n\n".join(SOURCE_PARAGRAPHS[3:]),
]

for page_content in pdf_pages_content:
    page = pdf_document.new_page()

    remaining_space = page.insert_textbox(
        pymupdf.Rect(50, 50, 545, 780), page_content, fontsize=11
    )

    if remaining_space < 0:
        raise RuntimeError("The PDF text does not fit inside the page.")

pdf_document.save(PDF_PATH)
pdf_document.close()


# ============================================================
# 7. Extract TXT Content
# ============================================================


def extract_txt(file_path: Path) -> str:

    return file_path.read_text(encoding="utf-8")


# ============================================================
# 8. Extract Markdown Content
# ============================================================


def extract_markdown(file_path: Path) -> str:

    return file_path.read_text(encoding="utf-8")


# ============================================================
# 9. Extract PDF Content Page by Page
# ============================================================


def extract_pdf(file_path: Path) -> list[dict]:

    extracted_pages = []

    with pymupdf.open(file_path) as document:
        for page_number, page in enumerate(document, start=1):
            page_text = page.get_text("text", sort=True)

            extracted_pages.append(
                {
                    "page_number": page_number,
                    "text": page_text,
                    "character_count": len(page_text),
                    "is_empty": not page_text.strip(),
                }
            )

    return extracted_pages


# ============================================================
# 10. Load All Documents
# ============================================================

txt_text = extract_txt(TXT_PATH)

markdown_text = extract_markdown(MD_PATH)

pdf_pages = extract_pdf(PDF_PATH)

pdf_text = "\n\n".join(page["text"] for page in pdf_pages)


# ============================================================
# 11. Calculate Text Statistics
# ============================================================


def calculate_statistics(text: str) -> dict:

    return {
        "characters": len(text),
        "words": len(text.split()),
        "lines": len(text.splitlines()),
    }


results = {
    "TXT": calculate_statistics(txt_text),
    "Markdown": calculate_statistics(markdown_text),
    "PDF": calculate_statistics(pdf_text),
}


# ============================================================
# 12. Normalize Whitespace for Comparison
# ============================================================


def normalize_whitespace(text: str) -> str:

    return re.sub(r"\s+", " ", text).strip()


normalized_source = normalize_whitespace(SOURCE_TEXT)

normalized_txt = normalize_whitespace(txt_text)

normalized_markdown = normalize_whitespace(markdown_text)

normalized_pdf = normalize_whitespace(pdf_text)


# ============================================================
# 13. Compare Extracted Content
# ============================================================

comparison = {
    "TXT_exact_match": (txt_text == SOURCE_TEXT),
    "TXT_normalized_match": (normalized_txt == normalized_source),
    "Markdown_contains_source": (SOURCE_TEXT in markdown_text),
    "Markdown_has_heading": (markdown_text.startswith("# ")),
    "PDF_normalized_match": (normalized_pdf == normalized_source),
    "PDF_page_count": len(pdf_pages),
}


# ============================================================
# 14. Display Results
# ============================================================

print("=" * 65)
print("DOCUMENT LOADING AND TEXT EXTRACTION")
print("=" * 65)


print("\n1. GENERATED DOCUMENTS")

for file_path in [TXT_PATH, MD_PATH, PDF_PATH]:
    print({"name": file_path.name, "size_bytes": file_path.stat().st_size})


print("\n2. EXTRACTION STATISTICS")

print(json.dumps(results, indent=4, ensure_ascii=False))


print("\n3. CONTENT COMPARISON")

print(json.dumps(comparison, indent=4))


print("\n4. PDF PAGE INFORMATION")

print(json.dumps(pdf_pages, indent=4, ensure_ascii=False))


print("\n5. TEXT PREVIEW")

print("TXT:")
print(txt_text[:150])

print("\nMARKDOWN:")
print(markdown_text[:150])

print("\nPDF:")
print(pdf_text[:150])


print("\nEXPERIMENT COMPLETED")

DOCUMENT LOADING AND TEXT EXTRACTION

1. GENERATED DOCUMENTS
{'name': 'sample.txt', 'size_bytes': 446}
{'name': 'sample.md', 'size_bytes': 477}
{'name': 'sample.pdf', 'size_bytes': 1636}

2. EXTRACTION STATISTICS
{
    "TXT": {
        "characters": 436,
        "words": 59,
        "lines": 11
    },
    "Markdown": {
        "characters": 465,
        "words": 64,
        "lines": 13
    },
    "PDF": {
        "characters": 436,
        "words": 59,
        "lines": 13
    }
}

3. CONTENT COMPARISON
{
    "TXT_exact_match": true,
    "TXT_normalized_match": true,
    "Markdown_contains_source": true,
    "Markdown_has_heading": true,
    "PDF_normalized_match": true,
    "PDF_page_count": 2
}

4. PDF PAGE INFORMATION
[
    {
        "page_number": 1,
        "text": "HADOTA Local AI Assistant\n\nHADOTA is a local AI assistant powered by Qwen through Ollama.\n\nThe application uses FastAPI as its backend framework.",
        "character_count": 145,
        "is_empty": false
    },
  

### Result Interpretation

Extraction Statistics

| Format | Characters | Words | Lines |
|---|---:|---:|---:|
| TXT | 436 | 59 | 11 |
| Markdown | 465 | 64 | 13 |
| PDF | 436 | 59 | 13 |

The TXT extractor reproduced the original source text exactly.

The Markdown extractor preserved the original content and its additional heading syntax. The character and word counts increased because the Markdown fixture contained an additional heading.

The PDF extractor successfully recovered the source content after whitespace normalization. However, the extracted line count differed because PDF text layout introduced additional line breaks.

The PDF extraction process also preserved page-level information, producing two separate page records.

### Finding

The experiment demonstrated that TXT, Markdown, and PDF require different extraction strategies.

TXT and Markdown can be loaded through standard Python file operations, while PDF requires a dedicated extraction library such as PyMuPDF.

Page-level extraction is important for preserving document provenance and supporting future source attribution.

Whitespace normalization can help compare extracted content, but it must not be confused with general-purpose text normalization.

### Limitations

The experiment used controlled document fixtures with simple textual content.

It did not evaluate scanned PDFs, OCR, complex tables, multi-column layouts, or extraction quality across heterogeneous real-world documents.